<a href="https://colab.research.google.com/github/nabilah-afrin/recommendation_system_rokomri_books/blob/secondary/rokomari_bn_books_modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/Dataset/rokomari_books/Rokomari Recommendation Dataset/Datasets

/content/drive/.shortcut-targets-by-id/1SdeIcOv6xY-c8y2UcMwPLYx_BaB0zdXx/Rokomari Recommendation Dataset/Datasets


In [ ]:
!ls

 corrected_language.csv        rokomari_book_data_v2.csv	   rokomari_v2.ipynb
'Data Analysis Report.gdoc'    rokomari_books_only_bangla_v2.csv  'scraping log.txt'
 mixed_title_rokomari_v2.csv  'Rokomari RS task-sheet.gsheet'	   wrong_language_url.txt
 rokomari_book_data.csv        rokomari_v2.csv


# 2. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from gensim.models import Word2Vec
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from gensim.utils import simple_preprocess
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 3. Data load

In [ ]:
df = pd.read_csv("rokomari_books_only_bangla_v2.csv")

In [ ]:
df.head()

,book_id,title,author,publisher,publisher_name_english,categories,category_english,edition,isbn,summary,...,book_url,prod_img_link,availability,product_category,actual_rating,bangla_title,language_title,language_categories,categories_fixed,language_author
0,350167,অমানুষিক,মনোরঞ্জন ব্যাপারী,একা,Eka (India),পশ্চিমবঙ্গের বই,West Bengal Books,6 March 2023,9789357762298,No summary,...,https://www.rokomari.com/book/350167/amanushik,https://img.cf.rokomari.com/ProductNew20190903...,Request For Reprint,book,0.0,অমানুষিক,bn,bn,পশ্চিমবঙ্গের বই,bn
1,377700,নির্বাচিত গল্প সংকলন,লু স্যুন,ছাড়পত্র প্রকাশন (ইন্ডিয়া),Charpatra Prakashan (India),পশ্চিমবঙ্গের বই: সমকালীন গল্প,West Bengal Books: Contemporary Story,Edition,9788194097303,No summary,...,https://www.rokomari.com/book/377700/nirbachit...,https://img.cf.rokomari.com/ProductNew20190903...,Request For Reprint,book,0.0,নির্বাচিত গল্প সংকলন,bn,bn,"পশ্চিমবঙ্গের বই, সমকালীন গল্প",bn
2,377718,তিমুর ও তার দলবল,আর্কাদি গাইদার,ছাড়পত্র প্রকাশন (ইন্ডিয়া),Charpatra Prakashan (India),পশ্চিমবঙ্গের বই: শিশু-কিশোর উপন্যাস,West Bengal Books: Children and Teens Novel,Edition,9788193860939,No summary,...,https://www.rokomari.com/book/377718/timur-o-t...,https://img.cf.rokomari.com/ProductNew20190903...,Request For Reprint,book,0.0,তিমুর ও তার দলবল,bn,bn,"পশ্চিমবঙ্গের বই, শিশু-কিশোর উপন্যাস",bn
3,189529,চেক ডিসঅনার মামলার সহজ ভাষ্য,মোঃ কাইছার হামিদ,এ.কে লিগ্যাল সল্যুশন,A.K Legal Solution,ব্যাংকিং এন্ড কমার্স ল,Banking and Commerce Law,1st Published,No ISBN,* চেক ডিসঅনার ও মামলা দায়েরের পদ্ধতি সংক্রান্...,...,https://www.rokomari.com/book/189529/cheque-di...,https://img.cf.rokomari.com/ProductNew20190903...,Not Available,book,5.0,চেক ডিসঅনার মামলার সহজ ভাষ্য,bn,bn,ব্যাংকিং এন্ড কমার্স ল,bn
4,325781,হাইকোর্ট এক্সাম ফর্মুলা,মোঃ কাইছার হামিদ,A.K Legal Solution,A.K Legal Solution,অ্যাডভোকেসি/বিচার আইন,Advocacy/ Adjudication Law,Edition,9789843545589,"""High Court Exam Formula"" book will be very he...",...,https://www.rokomari.com/book/325781/high-cour...,https://img.cf.rokomari.com/ProductNew20190903...,Request For Reprint,book,0.0,হাইকোর্ট এক্সাম ফর্মুলা,bn,bn,"অ্যাডভোকেসি, বিচার আইন",bn


## 3.1 Essential portion of the Data

In [ ]:
# let's just create a dataframe that we are going to use

# we'll take author, bangla_title, categories_fiexd, also in that order

df = df.loc[:, ['author', 'bangla_title', 'categories_fixed']]

# also, merge the author, bangla_title and categories_fixed to form a
# combined feature

df['combined_feature'] = df['author'] + ' ' + df['bangla_title'] + ' ' + df['categories_fixed']
# df['combined_feature'] = df['bangla_title'] + ' ' + df['categories_fixed']

In [ ]:
df['combined_feature'].head(5)

,combined_feature
0,মনোরঞ্জন ব্যাপারী অমানুষিক পশ্চিমবঙ্গের বই
1,"লু স্যুন নির্বাচিত গল্প সংকলন পশ্চিমবঙ্গের বই,..."
2,আর্কাদি গাইদার তিমুর ও তার দলবল পশ্চিমবঙ্গের ব...
3,মোঃ কাইছার হামিদ চেক ডিসঅনার মামলার সহজ ভাষ্য ...
4,মোঃ কাইছার হামিদ হাইকোর্ট এক্সাম ফর্মুলা অ্যাড...


In [ ]:
df.head()

,author,bangla_title,categories_fixed,combined_feature
0,মনোরঞ্জন ব্যাপারী,অমানুষিক,পশ্চিমবঙ্গের বই,মনোরঞ্জন ব্যাপারী অমানুষিক পশ্চিমবঙ্গের বই
1,লু স্যুন,নির্বাচিত গল্প সংকলন,"পশ্চিমবঙ্গের বই, সমকালীন গল্প","লু স্যুন নির্বাচিত গল্প সংকলন পশ্চিমবঙ্গের বই,..."
2,আর্কাদি গাইদার,তিমুর ও তার দলবল,"পশ্চিমবঙ্গের বই, শিশু-কিশোর উপন্যাস",আর্কাদি গাইদার তিমুর ও তার দলবল পশ্চিমবঙ্গের ব...
3,মোঃ কাইছার হামিদ,চেক ডিসঅনার মামলার সহজ ভাষ্য,ব্যাংকিং এন্ড কমার্স ল,মোঃ কাইছার হামিদ চেক ডিসঅনার মামলার সহজ ভাষ্য ...
4,মোঃ কাইছার হামিদ,হাইকোর্ট এক্সাম ফর্মুলা,"অ্যাডভোকেসি, বিচার আইন",মোঃ কাইছার হামিদ হাইকোর্ট এক্সাম ফর্মুলা অ্যাড...


In [ ]:
df.loc[0, 'combined_feature']

'মনোরঞ্জন ব্যাপারী অমানুষিক পশ্চিমবঙ্গের বই'

In [ ]:
import re
def remove_special_symbols(text):
    return re.sub(r'[ঃ:,-]', '', str(text))


In [ ]:
df['combined_feature'] = df.loc[:, 'combined_feature'].apply(remove_special_symbols)
# df['tokenized_feature'] = df.loc[:, 'combined_feature'].apply(simple_preprocess)
df['tokenized_feature'] = df['combined_feature'].apply(lambda x: x.split())

In [ ]:
df.head()

,author,bangla_title,categories_fixed,combined_feature,tokenized_feature
0,মনোরঞ্জন ব্যাপারী,অমানুষিক,পশ্চিমবঙ্গের বই,মনোরঞ্জন ব্যাপারী অমানুষিক পশ্চিমবঙ্গের বই,"[মনোরঞ্জন, ব্যাপারী, অমানুষিক, পশ্চিমবঙ্গের, বই]"
1,লু স্যুন,নির্বাচিত গল্প সংকলন,"পশ্চিমবঙ্গের বই, সমকালীন গল্প",লু স্যুন নির্বাচিত গল্প সংকলন পশ্চিমবঙ্গের বই ...,"[লু, স্যুন, নির্বাচিত, গল্প, সংকলন, পশ্চিমবঙ্গ..."
2,আর্কাদি গাইদার,তিমুর ও তার দলবল,"পশ্চিমবঙ্গের বই, শিশু-কিশোর উপন্যাস",আর্কাদি গাইদার তিমুর ও তার দলবল পশ্চিমবঙ্গের ব...,"[আর্কাদি, গাইদার, তিমুর, ও, তার, দলবল, পশ্চিমব..."
3,মোঃ কাইছার হামিদ,চেক ডিসঅনার মামলার সহজ ভাষ্য,ব্যাংকিং এন্ড কমার্স ল,মো কাইছার হামিদ চেক ডিসঅনার মামলার সহজ ভাষ্য ব...,"[মো, কাইছার, হামিদ, চেক, ডিসঅনার, মামলার, সহজ,..."
4,মোঃ কাইছার হামিদ,হাইকোর্ট এক্সাম ফর্মুলা,"অ্যাডভোকেসি, বিচার আইন",মো কাইছার হামিদ হাইকোর্ট এক্সাম ফর্মুলা অ্যাডভ...,"[মো, কাইছার, হামিদ, হাইকোর্ট, এক্সাম, ফর্মুলা,..."


In [ ]:
df = df.merge(df['author'].value_counts().reset_index(), on='author', how='inner')

In [ ]:
df.rename(columns={'count': 'author_count'}, inplace=True)

In [ ]:
print(f"Shape: {df.shape}")

Shape: (204572, 6)


# Word2vec model

In [ ]:
# Train Word2Vec model on the tokenized data
tokenized_books = df['tokenized_feature'].tolist()
wv_model = Word2Vec(sentences=tokenized_books, vector_size=100, window=4, min_count=1, workers=4)

# wv_model = Word2Vec(vector_size=100, window=5, min_count=1, workers=4)

In [ ]:
# wv_model.build_vocab(df['tokenized_feature'])

In [ ]:
# wv_model.corpus_count
# Train the model
# wv_model.train(tokenized_books, total_examples=wv_model.corpus_count, epochs=30)

In [ ]:
def average_word_vectors(words, model, vocabulary, num_features):
    feature_vector = np.zeros((num_features,), dtype="float64")
    nwords = 0.

    for word in words:
        if word in vocabulary:
            nwords = nwords + 1.
            feature_vector = np.add(feature_vector, model.wv[word])

    if nwords:
        feature_vector = np.divide(feature_vector, nwords)

    return feature_vector

# Function to compute average word vectors for all books
def averaged_word_vectorizer(corpus, model, num_features):
    vocabulary = set(model.wv.index_to_key)
    features = [average_word_vectors(tokenized_sentence, model, vocabulary, num_features) for tokenized_sentence in corpus]
    return np.array(features)

In [ ]:
# Compute average word vectors for all books
# w2v_feature_array = averaged_word_vectorizer(corpus=df['tokenized_feature'], model=wv_model, num_features=100)

In [ ]:
def get_book_vector(words, model):
    # Get vectors for each word, ignore words not in the model's vocabulary
    word_vectors = [model.wv[word] for word in words if word in model.wv]
    if len(word_vectors) == 0:
        return np.zeros(model.vector_size)  # Return zero vector if no words match
    return np.mean(word_vectors, axis=0)

# Compute vectors for all books
df['book_vector'] = df['tokenized_feature'].apply(lambda x: get_book_vector(x, wv_model))

In [ ]:
# Function to get similar books based on input text
def wv_similar_books(input_text, model=wv_model, topn=20):
    # Tokenize the input text
    input_tokens = input_text.split()
    # Get the vector for the input by averaging word vectors
    input_vector = get_book_vector(input_tokens, model)

    # Compute cosine similarity between input vector and all book vectors
    book_vectors = np.vstack(df['book_vector'].values)
    similarities = cosine_similarity([input_vector], book_vectors)[0]

    # Get top N most similar books
    df['similarity_score'] = similarities
    results = df[['author', 'bangla_title', 'categories_fixed', 'similarity_score']].sort_values(by='similarity_score', ascending=False).head(topn)

    return results

In [ ]:
wv_similar_books('দেয়াল')

,author,bangla_title,categories_fixed,similarity_score
8079,হাবীবুল্লাহ সিরাজী,গদ্যের গন্ধগোকুল,মুক্ত গদ্য,0.917239
48520,সাদ আমির,দূরের আকাশ হতে,মুক্ত গদ্য,0.910165
86447,সানজিদা হোসাইন,প্যাথেটিক ফ্যালাসি,মুক্ত গদ্য,0.909277
20502,আতিক ফারুক,এখানে আরেকটু রোদ,মুক্ত গদ্য,0.907001
20503,আতিক ফারুক,বুনোফুলের দিন,মুক্ত গদ্য,0.896761
72069,হাসনাইন মঞ্জুর মুর্শেদ,ইতস্ততঃ অস্তিত্ব,মুক্ত গদ্য,0.896075
20505,আতিক ফারুক,যেকোনো স্মৃতির পাশে,মুক্ত গদ্য,0.895835
24121,কানিজ ফাতেমা খুশী,পাতার ফাঁকে গলে প্রেম,মুক্ত গদ্য,0.895739
20506,আতিক ফারুক,মেহেরুন প্রিয়তম ফুল,মুক্ত গদ্য,0.892500
17581,রাহেল রাজিব,অনুমেয় আঘাতের ক্ষত,গদ্য,0.891795


In [ ]:
df[df['bangla_title'] == "হাঁস চলার পথ"].index[0]

3

In [ ]:
# Get the user input
user_book = input("Enter a movie title: ")

# Find the index of the user book
book_index = df[df['bangla_title'] == user_book].index[0]

In [ ]:
print(df.loc[:, 'combined_feature'][1])

আর্কাদি গাইদার তিমুর ও তার দলবল পশ্চিমবঙ্গের বই শিশুকিশোর উপন্যাস


## Fastetx

In [ ]:
model.wv.most_similar(positive=['dog'])


# 4. Using Doc2Vec

In [ ]:
tagged_data = [TaggedDocument(words=row.split(), tags=[str(i)]) for i, row in enumerate(df['combined_feature'])]

In [ ]:
dv_model = Doc2Vec(tagged_data, vector_size=100, window=4, min_count=1, workers=4, epochs=50)

In [ ]:
text = "পশ্চিমবঙ্গের বই, শিশু-কিশোর উপন্যাস"
input_vector = dv_model.infer_vector(text.split())
input_vector

array([ 0.05839917,  0.10673126, -0.17147042, -0.02709691,  0.01510327,
       -0.09941801,  0.11658876,  0.28679425, -0.15936676, -0.07802116,
       -0.03477606, -0.10255055, -0.04935367,  0.02507915, -0.10635126,
       -0.2527119 , -0.09189694, -0.12642328,  0.00715667, -0.08209583,
        0.13869926,  0.16047223,  0.22092763, -0.06891996,  0.02523756,
       -0.04178124, -0.06921628, -0.14707224,  0.01689284,  0.0413558 ,
        0.06786394,  0.11816014,  0.12199622,  0.12857884, -0.04122186,
        0.01298661, -0.09530645, -0.13031454, -0.05891198, -0.19534047,
        0.06539464, -0.03766237, -0.12282142, -0.10612173,  0.11751659,
        0.11664615, -0.11323731, -0.00917651,  0.12076712,  0.08108999,
        0.01539935, -0.06866321, -0.14833947,  0.01829575, -0.02384706,
       -0.00707383,  0.18345286,  0.05151917, -0.22149841,  0.0896749 ,
        0.02630426,  0.00211941, -0.07157642,  0.17937912, -0.08549369,
        0.08659296, -0.13562359,  0.28572696, -0.39104655,  0.21

In [ ]:
# Function to get similar books based on input text
def dv_similar_books(input_text, model=dv_model, topn=20):
    # Tokenize the input text
    input_tokens = input_text.split()
    # Get the vector for the input by averaging word vectors
    # input_vector = get_book_vector(input_tokens, model)
    input_vector = model.infer_vector(input_tokens)

    # Compute cosine similarity between input vector and all book vectors
    # book_vectors = np.vstack(df['book_vector'].values)
    # similarities = cosine_similarity([input_vector], book_vectors)[0]
    similarities = dv_model.dv.most_similar([input_vector], topn=topn)
    book_index = [int(similarity[0]) for similarity in similarities]

    # Get top N most similar books
    # df['similarity_score'] = similarities
    # results = df[['author', 'bangla_title', 'categories_fixed', 'similarity_score']].sort_values(by='similarity_score', ascending=False).head(topn)
    results = df.loc[book_index, ['author', 'bangla_title', 'categories_fixed']]
    return results

In [ ]:
dv_similar_books('দেয়াল')

,author,bangla_title,categories_fixed
68488,তাপস কুমার হাজরা,ঘুম কেড়ে নেওয়ার ভোরে,পশ্চিমবঙ্গের বই
81849,শারমিনা পারভিন,স্মৃতিতে মুক্তিযুদ্ধ ও আমার জীবন,"মুক্তিযুদ্ধের ডায়েরি, চিঠি, স্মৃতিচারণ"
26596,আহমাদ সাব্বির,কিংবদন্তির কথা বলছি,মুসলিম ব্যক্তিত্ব
87357,ওলাফ ক্যারো,দ্য পাঠানস,ইতিহাস সম্পর্কিত অনুবাদ বই
49878,ড অনুপম সেন,সুন্দরের বিচার সভাতে,বাংলা কবিতা
31911,আব্দুর রাহমান ইবনু সালিহ আলমাহমুদ,অপরিহার্য শরীয়াহ,"ইসলামি বিধি-বিধান, মাসআলা-মাসায়েল"
63445,মোজাম্মেল হক সজল,সজল স্যারের ভূতের ম্যাজিক,শিশু-কিশোর গল্প
3166,সোমনাথ দাস,শিক্ষা প্রযুক্তিবিজ্ঞান:,"পশ্চিমবঙ্গের বই, গণিত, বিজ্ঞান, প্রযুক্তি"
42136,জয়দেব কর,রুমির কথামঞ্জরি,"উক্তি, বাণী, শ্লোক, প্রবাদ-প্রবচন"
30670,শাহনাজ বেগম,জননী,সমকালীন গল্প


In [ ]:
print(f"Enter Book name: {text}\n")
for i in book_index:
    print(
        f"Author: {df.loc[i, 'author']}\n"\
        f"Title: {df.loc[i, 'bangla_title']}\n"\
        f"Genre: {df.loc[i, 'categories_fixed']}\n"
        )

Enter Book name: পশ্চিমবঙ্গের বই, শিশু-কিশোর উপন্যাস

Author: বিভূতিভূষণ বন্দ্যোপাধ্যায়
Title: দেবযান
Genre: পশ্চিমবঙ্গের বই, উপন্যাস

Author: বিভূতিভূষণ বন্দ্যোপাধ্যায়
Title: দৃষ্টিপ্রদীপ
Genre: পশ্চিমবঙ্গের বই, উপন্যাস

Author: সমরেশ মজুমদার
Title: আগুনবেলা
Genre: পশ্চিমবঙ্গের বই, উপন্যাস

Author: ইমদাদুল হক মিলন
Title: অধিবাস
Genre: পশ্চিমবঙ্গের উপন্যাস

Author: মানবেন্দ্র মুখোপাধ্যায়
Title: গোষ্ঠিজীবনের উপন্যাস
Genre: পশ্চিমবঙ্গের বই

Author: আদিত্য মুখোপাধ্যায়
Title: শিখণ্ডী
Genre: পশ্চিমবঙ্গের উপন্যাস

Author: স্বপ্নময় চক্রবর্তী
Title: অবন্তীনগর
Genre: পশ্চিমবঙ্গের বই, উপন্যাস

Author: অনুপম মুখোপাধ্যায়
Title: খ্রিস্ট
Genre: পশ্চিমবঙ্গের বই, উপন্যাস

Author: মৃগাঙ্ক ভট্টাচার্য
Title: বসন্তপথ
Genre: পশ্চিমবঙ্গের বই, উপন্যাস

Author: ফাল্গুনী মুখোপাধ্যায়
Title: মনময়ূরী
Genre: পশ্চিমবঙ্গের বই, উপন্যাস



In [ ]:
import difflib
text = 'দয়ল'
difflib.get_close_matches(text, df['bangla_title'].tolist(), n=5)



['দোয়েল', 'দেয়াল', 'দেয়াল', 'দেয়াল', 'বদল']

In [ ]:
def retrieve_matched_entries(dataframe, input_text, title=True):
    close_matches = difflib.get_close_matches(input_text,
                                              dataframe['bangla_title'].tolist() + dataframe['author'].tolist(),
                                              n=50,
                                              cutoff=0.4)

    condition1 = dataframe['bangla_title'].isin(close_matches)
    condition2 = dataframe['author'].isin(close_matches)
    # matched_titles = dataframe.loc[(condition1) | (condition2), ['author', 'bangla_title', 'categories_fixed']]
    matched_titles = dataframe.loc[condition1, ['author', 'bangla_title', 'categories_fixed', 'author_count']]
    matched_author = dataframe.loc[condition2, ['author', 'bangla_title', 'categories_fixed', 'author_count']]

    if title:
        matched_entries = pd.concat([matched_titles, matched_author])
        return matched_entries.sort_values(by='author_count', ascending=False)

    matched_entries = pd.concat([matched_author, matched_titles])
    return matched_entries.sort_values(by='author_count', ascending=False)


In [ ]:
text = 'হুমায়ুন আহমেদ'
retrieved_elem = retrieve_matched_entries(df, text, title=False).sample(20).reset_index(drop=True)

In [ ]:
retrieved_elem

,author,bangla_title,categories_fixed,author_count
0,হুমায়ূন আহমেদ,মিসির আলি অমনিবাস১,সমকালীন উপন্যাস,276
1,হুমায়ূন আহমেদ,ময়ূরাক্ষী,সমকালীন উপন্যাস,276
2,হুমায়ূন আহমেদ,জোছনা ও জননীর গল্প,"রাজনৈতিক, মুক্তিযুদ্ধভিত্তিক উপন্যাস",276
3,হুমায়ূন আহমেদ,পোকা,সমকালীন উপন্যাস,276
4,হুমায়ূন আহমেদ,মিসির আলির চশমা,সমকালীন উপন্যাস,276
5,হুমায়ূন আহমেদ,অয়োময়,সমকালীন গল্প,276
6,হুমায়ূন আহমেদ,হিমু মামা,শিশু-কিশোর উপন্যাস,276
7,হুমায়ূন আহমেদ,হুমায়ূন আহমেদ রচনাবলী ৪,"রচনা সংকলন, সমগ্র",276
8,হুমায়ূন আহমেদ,হিমু রিমান্ডে,সমকালীন উপন্যাস,276
9,হুমায়ূন আহমেদ,সে ও নর্তকী,সমকালীন উপন্যাস,276


In [ ]:
get_cat = retrieved_elem.loc[0]

In [ ]:
get_cat

,0
author,হুমায়ূন আহমেদ
bangla_title,মিসির আলি অমনিবাস১
categories_fixed,সমকালীন উপন্যাস
author_count,276


In [ ]:

text = get_cat['author'] + " " + get_cat['bangla_title'] + " " + get_cat['categories_fixed']
dv_similar_books(text)

,author,bangla_title,categories_fixed
45525,সুদর্শন দাস,নাট্যগুচ্ছ,"পশ্চিমবঙ্গের বই, নাটক"
2049,হালিমা খাতুন,কিশোর ভুবন ছোট গল্প,শিশু-কিশোর গল্প
85976,শামীম ইসলাম,জীবনের জয় ও অমর বাণী,"উক্তি, বাণী, শ্লোক, প্রবাদ-প্রবচন"
19158,অলকা সরকার কেয়া,বঙ্গবন্ধু তোমার নামে,"মুক্তিযুদ্ধ, ভাষা আন্দোলন, রাজনৈতিক কবিতা"
2185,ধ্রুব এষ,মনে পড়ে এই হেমন্তের রাতে,সমকালীন উপন্যাস
81365,মুহম্মদ মোকাররম হোসায়েন,বিবিক্ত,বাংলা কবিতা
42883,অমিভাত হালদার সম্পাদক,আমি কিংবদন্তি হব,বাংলা কবিতা
35456,ভীষ্মদেব চৌধুরী,জনান্তিকের মুক্তিযুদ্ধ,"সমাজ, সভ্যতা, সংস্কৃতি বিষয়ক প্রবন্ধ"
7392,মঈনুল আহসান সাবের,মানুষ যেখানে যায় না,সমকালীন উপন্যাস
86335,গাজী শরিফুল হাসান,বার্মিংহাম ডায়েরি,ভ্রমণ বিষয়ক স্মৃতি


In [ ]:
text = get_cat['author'] + " " + get_cat['bangla_title'] + " " + get_cat['categories_fixed']
# text = get_cat['categories_fixed']
wv_similar_books(text)[1:]

,author,bangla_title,categories_fixed,similarity_score
11517,হুমায়ূন আহমেদ,মিসির আলি অমনিবাস২,সমকালীন উপন্যাস,0.999995
11574,হুমায়ূন আহমেদ,মিসির আলি আনসলভ্ড,সমকালীন উপন্যাস,0.999992
11627,হুমায়ূন আহমেদ,বাঘবন্দি মিসির আলি,সমকালীন উপন্যাস,0.999992
11649,হুমায়ূন আহমেদ,হিমু মিসির আলি যুগলবন্দি,সমকালীন উপন্যাস,0.999613
11403,হুমায়ূন আহমেদ,মিসির আলির চশমা,সমকালীন উপন্যাস,0.998339
11412,হুমায়ূন আহমেদ,একজন মায়াবতী,সমকালীন উপন্যাস,0.997362
11424,হুমায়ূন আহমেদ,নন্দিত নরকে,সমকালীন উপন্যাস,0.997304
11419,হুমায়ূন আহমেদ,মাতাল হাওয়া,সমকালীন উপন্যাস,0.997107
11629,হুমায়ূন আহমেদ,এইসব দিনরাত্রি,সমকালীন উপন্যাস,0.997029
11566,হুমায়ূন আহমেদ,মীরার গ্রামের বাড়ী,সমকালীন উপন্যাস,0.996825


In [ ]:
df.columns

Index(['author', 'bangla_title', 'categories_fixed', 'combined_feature',
       'tokenized_feature', 'book_vector', 'similarity_score', 'count'],
      dtype='object')